# Khám phá dataset văn bản pháp luật

Notebook này thống kê các văn bản trong `data/dataset`: số package, số file, loại file chính/phụ lục/mẫu/QCVN, dung lượng, các trường hợp đáng kiểm tra và tình trạng output đã parse trong `data/preprocessed/parsed` nếu có.

Notebook dùng standard library là chính, nên có thể chạy ngay cả khi môi trường chưa cài `pandas` hoặc `matplotlib`.


In [11]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
import csv
import html
import json
import re
import statistics
import unicodedata

DATASET_DIR = Path('data/dataset')
DATASET_DIR_1 = Path(r'\\wsl.localhost\Ubuntu-22.04\home\vinh\projects\traffic-bot\data\dataset')
PARSED_DIR = Path('data/preprocessed/parsed')

assert DATASET_DIR.exists(), f'Không tìm thấy thư mục: {DATASET_DIR.resolve()}'
print('Dataset:', DATASET_DIR.resolve())
print('Parsed output:', PARSED_DIR.resolve())


Dataset: D:\Users\ADMIN\LocalData\Classworks\KLTN\data\dataset
Parsed output: D:\Users\ADMIN\LocalData\Classworks\KLTN\data\preprocessed\parsed


## Hàm tiện ích


In [2]:
try:
    from legal_parser_modular.legal_parser.common.file_classifier import (
        is_probable_attachment,
        is_probable_main_document,
    )
except Exception:
    is_probable_attachment = None
    is_probable_main_document = None


def strip_accents(text: str) -> str:
    text = (text or '').replace('đ', 'd').replace('Đ', 'D')
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')


def normalize_name(text: str) -> str:
    text = strip_accents(text or '').lower()
    text = re.sub(r'[_\-]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def classify_file(path: Path) -> str:
    name = normalize_name(path.stem)
    if path.suffix.lower() not in {'.docx', '.doc'}:
        return 'other'
    if is_probable_attachment and is_probable_attachment(path):
        if re.search(r'\bqcvn\b|quy\s*chuan', name):
            return 'qcvn'
        if re.search(r'\bmau\b|\bdkx\b', name):
            return 'form'
        return 'appendix'
    if is_probable_main_document and is_probable_main_document(path):
        return 'main_candidate'
    if re.search(r'\b(phu\s*luc|mau|dkx|qcvn|quy\s*chuan)\b', name):
        if re.search(r'\bqcvn\b|quy\s*chuan', name):
            return 'qcvn'
        if re.search(r'\bmau\b|\bdkx\b', name):
            return 'form'
        return 'appendix'
    if re.search(r'\b(luat|bo luat|nghi dinh|thong tu|quyet dinh|nghi quyet|doc)\b', name):
        return 'main_candidate'
    return 'unknown_doc'


def count_jsonl(path: Path) -> int | None:
    if not path.exists():
        return None
    with path.open('r', encoding='utf-8') as f:
        return sum(1 for line in f if line.strip())


def mb(size_bytes: int) -> float:
    return round(size_bytes / (1024 * 1024), 3)


def display_table(rows, columns=None, limit=20):
    rows = list(rows)
    if not rows:
        print('(empty)')
        return
    columns = columns or list(rows[0].keys())
    visible = rows[:limit]
    try:
        from IPython.display import HTML, display
        head = ''.join(f'<th>{html.escape(str(c))}</th>' for c in columns)
        body = []
        for row in visible:
            body.append('<tr>' + ''.join(f'<td>{html.escape(str(row.get(c, "")))}</td>' for c in columns) + '</tr>')
        note = f'<p><em>Hiển thị {len(visible)}/{len(rows)} dòng</em></p>' if len(rows) > len(visible) else ''
        display(HTML(note + '<table><thead><tr>' + head + '</tr></thead><tbody>' + ''.join(body) + '</tbody></table>'))
    except Exception:
        print(f'Hiển thị {len(visible)}/{len(rows)} dòng')
        print('\t'.join(columns))
        for row in visible:
            print('\t'.join(str(row.get(c, '')) for c in columns))


def counter_rows(counter: Counter, key_name='name'):
    return [{key_name: key, 'count': value} for key, value in counter.most_common()]


def text_bar(counter: Counter, width=40):
    if not counter:
        print('(empty)')
        return
    max_count = max(counter.values())
    for key, value in counter.most_common():
        bar = '█' * max(1, round(value / max_count * width))
        print(f'{str(key):<24} {value:>5} {bar}')


## Lập inventory file


In [3]:
files = []
for path in sorted(DATASET_DIR.rglob('*')):
    if not path.is_file() or path.name.startswith('~$'):
        continue
    rel = path.relative_to(DATASET_DIR)
    package_id = rel.parts[0] if len(rel.parts) > 1 else '(root)'
    stat = path.stat()
    files.append({
        'package_id': package_id,
        'file_name': path.name,
        'relative_path': str(rel),
        'extension': path.suffix.lower(),
        'kind': classify_file(path),
        'size_mb': mb(stat.st_size),
        'modified_at': datetime.fromtimestamp(stat.st_mtime).isoformat(sep=' ', timespec='seconds'),
    })

display_table(files, limit=10)


package_id,file_name,relative_path,extension,kind,size_mb,modified_at
100_2015_QH13,Bộ-luật-100-2015-QH13.docx,100_2015_QH13\Bộ-luật-100-2015-QH13.docx,.docx,main_candidate,0.258,2026-05-15 03:39:44
118_2025_QH15,Luật-118-2025-QH15.docx,118_2025_QH15\Luật-118-2025-QH15.docx,.docx,main_candidate,0.054,2026-05-11 07:23:50
119_2024_NDCP,Nghị định 119_2024_NĐ-CP DOC.docx,119_2024_NDCP\Nghị định 119_2024_NĐ-CP DOC.docx,.docx,main_candidate,0.075,2026-05-11 17:02:28
119_2024_NDCP,Phu luc. Thong tin trong CSDL thanh toan dien tu giao thong duong bo.docx,119_2024_NDCP\Phu luc. Thong tin trong CSDL thanh toan dien tu giao thong duong bo.docx,.docx,appendix,0.037,2026-05-11 17:02:28
12_2017_QH14,Luật-12-2017-QH14.docx,12_2017_QH14\Luật-12-2017-QH14.docx,.docx,main_candidate,0.122,2026-05-15 03:53:48
12_2025_TTBCA,Mau so 01. Ban de nghi xac minh giay phep lai xe.docx,12_2025_TTBCA\Mau so 01. Ban de nghi xac minh giay phep lai xe.docx,.docx,form,0.015,2026-05-11 21:55:30
12_2025_TTBCA,Mau so 01. Bao cao chung ket qua ky sat hach lai xe.docx,12_2025_TTBCA\Mau so 01. Bao cao chung ket qua ky sat hach lai xe.docx,.docx,form,0.017,2026-05-11 21:55:30
12_2025_TTBCA,Mau so 01. Bien ban kiem tra ho so va dieu kien du thi cua thi sinh du sat hach.docx,12_2025_TTBCA\Mau so 01. Bien ban kiem tra ho so va dieu kien du thi cua thi sinh du sat hach.docx,.docx,form,0.017,2026-05-11 21:55:30
12_2025_TTBCA,Mau so 01. Bien ban tong hop ket qua sat hach lai xe doi voi thi sinh du sat hach cac hang.docx,12_2025_TTBCA\Mau so 01. Bien ban tong hop ket qua sat hach lai xe doi voi thi sinh du sat hach cac hang.docx,.docx,form,0.018,2026-05-11 21:55:30
12_2025_TTBCA,Mau so 02. Ban xac minh giay phep lai xe.docx,12_2025_TTBCA\Mau so 02. Ban xac minh giay phep lai xe.docx,.docx,form,0.015,2026-05-11 21:55:30


## Tổng quan nhanh


In [4]:
package_dirs = sorted([p for p in DATASET_DIR.iterdir() if p.is_dir()])
root_files = [p for p in DATASET_DIR.iterdir() if p.is_file()]

summary = [
    {'metric': 'package folders', 'value': len(package_dirs)},
    {'metric': 'root-level files', 'value': len(root_files)},
    {'metric': 'total files', 'value': len(files)},
    {'metric': 'docx files', 'value': sum(1 for row in files if row['extension'] == '.docx')},
    {'metric': 'legacy doc files', 'value': sum(1 for row in files if row['extension'] == '.doc')},
    {'metric': 'total size MB', 'value': round(sum(row['size_mb'] for row in files), 3)},
]
display_table(summary, limit=20)

print('\nTheo extension:')
display_table(counter_rows(Counter(row['extension'] for row in files), 'extension'))

print('\nTheo loại file:')
display_table(counter_rows(Counter(row['kind'] for row in files), 'kind'))


metric,value
package folders,57
root-level files,0
total files,723
docx files,723
legacy doc files,0
total size MB,127.79



Theo extension:


extension,count
.docx,723



Theo loại file:


kind,count
form,515
appendix,135
main_candidate,47
qcvn,16
unknown_doc,10


## Thống kê theo package


In [5]:
files_by_package = defaultdict(list)
for row in files:
    files_by_package[row['package_id']].append(row)

packages = []
for package_dir in package_dirs:
    package_id = package_dir.name
    subset = files_by_package.get(package_id, [])
    kind_counter = Counter(row['kind'] for row in subset)
    ext_counter = Counter(row['extension'] for row in subset)
    parsed_pkg = PARSED_DIR / package_id
    packages.append({
        'package_id': package_id,
        'file_count': len(subset),
        'docx_count': ext_counter.get('.docx', 0),
        'doc_count': ext_counter.get('.doc', 0),
        'main_candidates': kind_counter.get('main_candidate', 0),
        'appendix_files': kind_counter.get('appendix', 0),
        'form_files': kind_counter.get('form', 0),
        'qcvn_files': kind_counter.get('qcvn', 0),
        'unknown_doc_files': kind_counter.get('unknown_doc', 0),
        'size_mb': round(sum(row['size_mb'] for row in subset), 3),
        'has_parsed_output': parsed_pkg.exists(),
        'parsed_all_units': count_jsonl(parsed_pkg / 'all_units.jsonl'),
        'parsed_all_refs': count_jsonl(parsed_pkg / 'all_ref_mentions.jsonl'),
    })

packages_sorted = sorted(packages, key=lambda row: (-row['file_count'], row['package_id']))
display_table(packages_sorted, limit=20)


package_id,file_count,docx_count,doc_count,main_candidates,appendix_files,form_files,qcvn_files,unknown_doc_files,size_mb,has_parsed_output,parsed_all_units,parsed_all_refs
158_2024_NDCP,88,88,0,1,0,87,0,0,14.568,True,None,None
47_2024_TTBGTVT,39,39,0,1,2,36,0,0,3.712,True,None,None
13_2025_TTBCA,37,37,0,1,0,36,0,0,4.149,True,None,None
14_2025_TTBXD,37,37,0,1,6,30,0,0,1.029,True,None,None
82_2024_TTBCA,34,34,0,1,5,28,0,0,3.822,True,None,None
160_2024_NDCP,29,29,0,1,2,26,0,0,1.0,True,None,None
165_2024_NDCP,29,29,0,1,4,24,0,0,0.703,True,None,None
12_2025_TTBCA,27,27,0,1,11,15,0,0,3.627,True,None,None
46_2024_TTBGTVT,27,27,0,1,1,25,0,0,0.666,True,None,None
69_2024_TTBCA,24,24,0,1,0,23,0,0,3.583,True,None,None


In [6]:
def numeric_summary(rows, field):
    values = [row[field] for row in rows if isinstance(row.get(field), (int, float))]
    if not values:
        return {'field': field, 'count': 0, 'min': None, 'median': None, 'mean': None, 'max': None}
    return {
        'field': field,
        'count': len(values),
        'min': min(values),
        'median': round(statistics.median(values), 3),
        'mean': round(statistics.mean(values), 3),
        'max': max(values),
    }

display_table([
    numeric_summary(packages, 'file_count'),
    numeric_summary(packages, 'docx_count'),
    numeric_summary(packages, 'main_candidates'),
    numeric_summary(packages, 'appendix_files'),
    numeric_summary(packages, 'size_mb'),
    numeric_summary(packages, 'parsed_all_units'),
], limit=20)


field,count,min,median,mean,max
file_count,57,1,10,12.684,88
docx_count,57,1,10,12.684,88
main_candidates,57,0,1,0.825,1
appendix_files,57,0,1,2.368,15
size_mb,57,0.016,0.657,2.242,35.903
parsed_all_units,0,None,None,None,None


## Các trường hợp nên kiểm tra thủ công


In [7]:
checks = {
    'package_no_docx': [row for row in packages if row['docx_count'] == 0],
    'package_no_main_candidate': [row for row in packages if row['docx_count'] > 0 and row['main_candidates'] == 0],
    'package_multiple_main_candidates': [row for row in packages if row['main_candidates'] > 1],
    'package_has_legacy_doc': [row for row in packages if row['doc_count'] > 0],
    'package_has_unknown_doc': [row for row in packages if row['unknown_doc_files'] > 0],
    'package_missing_parsed_output': [row for row in packages if not row['has_parsed_output']],
}

for name, rows in checks.items():
    print(f'\n{name}: {len(rows)}')
    display_table(sorted(rows, key=lambda row: row['package_id']), limit=30)



package_no_docx: 0
(empty)

package_no_main_candidate: 10


package_id,file_count,docx_count,doc_count,main_candidates,appendix_files,form_files,qcvn_files,unknown_doc_files,size_mb,has_parsed_output,parsed_all_units,parsed_all_refs
41_2024_TTBGTVT,15,15,0,0,3,11,0,1,0.352,True,None,None
49_2024_TTBGTVT,2,2,0,0,0,0,1,1,1.618,True,None,None
50_2024_TTBGTVT,3,3,0,0,0,0,2,1,2.834,True,None,None
52_2024_TTBGTVT,2,2,0,0,0,1,0,1,0.061,True,None,None
54_2024_TTBGTVT,22,22,0,0,5,15,1,1,0.657,True,None,None
69_2024_TTBQP,13,13,0,0,3,9,0,1,7.726,True,None,None
71_2024_TTBCA,5,5,0,0,4,0,0,1,1.404,True,None,None
72_2024_TTBCA,18,18,0,0,0,17,0,1,0.678,True,None,None
81_2024_TTBCA,2,2,0,0,0,0,1,1,1.674,True,None,None
83_2024_TTBCA,2,2,0,0,0,1,0,1,0.06,True,None,None



package_multiple_main_candidates: 0
(empty)

package_has_legacy_doc: 0
(empty)

package_has_unknown_doc: 10


package_id,file_count,docx_count,doc_count,main_candidates,appendix_files,form_files,qcvn_files,unknown_doc_files,size_mb,has_parsed_output,parsed_all_units,parsed_all_refs
41_2024_TTBGTVT,15,15,0,0,3,11,0,1,0.352,True,None,None
49_2024_TTBGTVT,2,2,0,0,0,0,1,1,1.618,True,None,None
50_2024_TTBGTVT,3,3,0,0,0,0,2,1,2.834,True,None,None
52_2024_TTBGTVT,2,2,0,0,0,1,0,1,0.061,True,None,None
54_2024_TTBGTVT,22,22,0,0,5,15,1,1,0.657,True,None,None
69_2024_TTBQP,13,13,0,0,3,9,0,1,7.726,True,None,None
71_2024_TTBCA,5,5,0,0,4,0,0,1,1.404,True,None,None
72_2024_TTBCA,18,18,0,0,0,17,0,1,0.678,True,None,None
81_2024_TTBCA,2,2,0,0,0,0,1,1,1.674,True,None,None
83_2024_TTBCA,2,2,0,0,0,1,0,1,0.06,True,None,None



package_missing_parsed_output: 0
(empty)


## File đáng chú ý


In [8]:
print('File lớn nhất:')
display_table(sorted(files, key=lambda row: row['size_mb'], reverse=True), limit=20)

print('\nFile nhỏ bất thường (< 10 KB):')
display_table(sorted([row for row in files if row['size_mb'] < 0.01], key=lambda row: row['size_mb']), limit=50)

print('\nFile chưa phân loại rõ:')
display_table(sorted([row for row in files if row['kind'] in {'unknown_doc', 'other'}], key=lambda row: (row['package_id'], row['file_name'])), limit=100)


File lớn nhất:


package_id,file_name,relative_path,extension,kind,size_mb,modified_at
51_2024_TTBGTVT,Thông-tư-51-2024-TT-BGTVT.docx,51_2024_TTBGTVT\Thông-tư-51-2024-TT-BGTVT.docx,.docx,main_candidate,35.903,2026-05-11 07:46:44
161_2024_NDCP,Nghị định 161_2024_NĐ-CP DOC.docx,161_2024_NDCP\Nghị định 161_2024_NĐ-CP DOC.docx,.docx,main_candidate,4.948,2026-05-11 16:57:34
161_2024_NDCP,Phu luc III. Mau nhan_bieu trung hang hoa nguy hiem.docx,161_2024_NDCP\Phu luc III. Mau nhan_bieu trung hang hoa nguy hiem.docx,.docx,form,4.521,2026-05-11 16:57:36
158_2024_NDCP,Nghị định 158_2024_NĐ-CP DOC.docx,158_2024_NDCP\Nghị định 158_2024_NĐ-CP DOC.docx,.docx,main_candidate,4.432,2026-05-11 16:59:28
53_2024_TTBGTVT,Thông tư 53_2024_TT-BGTVT DOC.docx,53_2024_TTBGTVT\Thông tư 53_2024_TT-BGTVT DOC.docx,.docx,main_candidate,2.497,2026-05-11 21:45:28
69_2024_TTBQP,69_2024_TT-BQP.docx,69_2024_TTBQP\69_2024_TT-BQP.docx,.docx,unknown_doc,2.395,2026-05-11 21:39:52
12_2025_TTBXD,Thông tư 12_2025_TT-BXD DOC.docx,12_2025_TTBXD\Thông tư 12_2025_TT-BXD DOC.docx,.docx,main_candidate,2.264,2026-05-11 21:55:08
13_2025_TTBCA,Thông tư 13_2025_TT-BCA DOC.docx,13_2025_TTBCA\Thông tư 13_2025_TT-BCA DOC.docx,.docx,main_candidate,1.85,2026-05-11 21:54:40
82_2024_TTBCA,Thông tư 82_2024_TT-BCA DOC.docx,82_2024_TTBCA\Thông tư 82_2024_TT-BCA DOC.docx,.docx,main_candidate,1.74,2026-05-11 21:35:34
47_2024_TTBGTVT,Thông tư 47_2024_TT-BGTVT DOC.docx,47_2024_TTBGTVT\Thông tư 47_2024_TT-BGTVT DOC.docx,.docx,main_candidate,1.658,2026-05-11 21:48:02



File nhỏ bất thường (< 10 KB):
(empty)

File chưa phân loại rõ:


package_id,file_name,relative_path,extension,kind,size_mb,modified_at
41_2024_TTBGTVT,41_2024_TT-BGTVT.docx,41_2024_TTBGTVT\41_2024_TT-BGTVT.docx,.docx,unknown_doc,0.111,2026-05-11 21:49:36
49_2024_TTBGTVT,49_2024_TT-BGTVT.docx,49_2024_TTBGTVT\49_2024_TT-BGTVT.docx,.docx,unknown_doc,0.82,2026-05-11 21:47:40
50_2024_TTBGTVT,50_2024_TT-BGTVT.docx,50_2024_TTBGTVT\50_2024_TT-BGTVT.docx,.docx,unknown_doc,1.415,2026-05-11 21:47:12
52_2024_TTBGTVT,52_2024_TT-BGTVT.docx,52_2024_TTBGTVT\52_2024_TT-BGTVT.docx,.docx,unknown_doc,0.043,2026-05-11 21:45:40
54_2024_TTBGTVT,54_2024_TT-BGTVT.docx,54_2024_TTBGTVT\54_2024_TT-BGTVT.docx,.docx,unknown_doc,0.2,2026-05-11 21:44:56
69_2024_TTBQP,69_2024_TT-BQP.docx,69_2024_TTBQP\69_2024_TT-BQP.docx,.docx,unknown_doc,2.395,2026-05-11 21:39:52
71_2024_TTBCA,71_2024_TT-BCA.docx,71_2024_TTBCA\71_2024_TT-BCA.docx,.docx,unknown_doc,0.705,2026-05-11 21:39:02
72_2024_TTBCA,72_2024_TT-BCA.docx,72_2024_TTBCA\72_2024_TT-BCA.docx,.docx,unknown_doc,0.254,2026-05-11 21:37:52
81_2024_TTBCA,81_2024_TT-BCA.docx,81_2024_TTBCA\81_2024_TT-BCA.docx,.docx,unknown_doc,0.838,2026-05-11 21:36:08
83_2024_TTBCA,83_2024_TT-BCA.docx,83_2024_TTBCA\83_2024_TT-BCA.docx,.docx,unknown_doc,0.045,2026-05-11 21:29:54


## Biểu đồ text nhanh


In [9]:
print('Số file theo loại:')
text_bar(Counter(row['kind'] for row in files))

print('\nTop package nhiều file nhất:')
text_bar(Counter({row['package_id']: row['file_count'] for row in packages_sorted[:20]}))


Số file theo loại:
form                       515 ████████████████████████████████████████
appendix                   135 ██████████
main_candidate              47 ████
qcvn                        16 █
unknown_doc                 10 █

Top package nhiều file nhất:
158_2024_NDCP               88 ████████████████████████████████████████
47_2024_TTBGTVT             39 ██████████████████
13_2025_TTBCA               37 █████████████████
14_2025_TTBXD               37 █████████████████
82_2024_TTBCA               34 ███████████████
160_2024_NDCP               29 █████████████
165_2024_NDCP               29 █████████████
12_2025_TTBCA               27 ████████████
46_2024_TTBGTVT             27 ████████████
69_2024_TTBCA               24 ███████████
54_2024_TTBGTVT             22 ██████████
79_2024_TTBCA               21 ██████████
36_2024_TTBGTVT             19 █████████
72_2024_TTBCA               18 ████████
12_2025_TTBXD               17 ████████
55_2024_TTBGTVT             17 ████████
16

## Xem thử nội dung đầu file DOCX

Cell này dùng `python-docx` để đọc vài đoạn đầu của một file bất kỳ. Đổi `sample_path` nếu muốn xem văn bản khác.


In [10]:
def docx_preview(path: Path, max_paragraphs: int = 12) -> list[str]:
    try:
        import docx
    except ImportError as exc:
        raise ImportError('Cần cài python-docx để đọc preview DOCX') from exc
    doc = docx.Document(str(path))
    out = []
    for p in doc.paragraphs:
        text = re.sub(r'\s+', ' ', p.text or '').strip()
        if text:
            out.append(text)
        if len(out) >= max_paragraphs:
            break
    return out

docx_files = sorted([row for row in files if row['extension'] == '.docx'], key=lambda row: (row['package_id'], row['kind'], row['file_name']))
if not docx_files:
    print('Không có file DOCX để preview')
else:
    sample_path = DATASET_DIR / docx_files[0]['relative_path']
    print(sample_path)
    for line in docx_preview(sample_path):
        print('-', line)


data\dataset\100_2015_QH13\Bộ-luật-100-2015-QH13.docx
- BỘ LUẬT
- HÌNH SỰ
- Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam;
- Quốc hội ban hành Bộ luật hình sự.
- Phần thứ nhất
- NHỮNG QUY ĐỊNH CHUNG
- Chương I
- ĐIỀU KHOẢN CƠ BẢN
- Điều 1. Nhiệm vụ của Bộ luật hình sự
- Bộ luật hình sự có nhiệm vụ bảo vệ chủ quyền quốc gia, an ninh của đất nước, bảo vệ chế độ xã hội chủ nghĩa, quyền con người, quyền công dân, bảo vệ quyền bình đẳng giữa đồng bào các dân tộc, bảo vệ lợi ích của Nhà nước, tổ chức, bảo vệ trật tự pháp luật, chống mọi hành vi phạm tội; giáo dục mọi người ý thức tuân theo pháp luật, phòng ngừa và đấu tranh chống tội phạm.
- Bộ luật này quy định về tội phạm và hình phạt.
- Điều 2. Cơ sở của trách nhiệm hình sự


## Xuất bảng thống kê ra CSV nếu cần


In [ ]:
def write_csv(path: Path, rows: list[dict]):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text('', encoding='utf-8-sig')
        return
    columns = list(rows[0].keys())
    with path.open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)


OUT_DIR = Path('data/preprocessed/reports')
write_csv(OUT_DIR / 'dataset_file_inventory.csv', files)
write_csv(OUT_DIR / 'dataset_package_summary.csv', packages)

print('Đã ghi:')
print('-', OUT_DIR / 'dataset_file_inventory.csv')
print('-', OUT_DIR / 'dataset_package_summary.csv')
